# TrialSpec-LM: Kaggle experiment runner

Thin wrapper. All logic lives in the GitHub repo; each experiment is one YAML in `configs/`.
`scripts/kaggle_pipeline.sh` runs: tests → smoke train → smoke predict → *(only if all pass)* zero-shot baseline → full training → fine-tuned val predictions.

**Settings:** Accelerator = GPU T4 x2, Internet = On, Secrets: `WANDB_API_KEY`, `HF_TOKEN`.

**New experiment = change `CONFIG` below, then Save Version → Save & Run All.** No other edits needed.

In [ ]:
CONFIG = 'configs/run2.yaml'
EXTRA_ARGS = ''        # '--smoke-only' to stop after smoke tests, '--skip-baseline' to skip zero-shot eval
BRANCH = 'master'

In [ ]:
import os, subprocess, time

def run(cmd):
    """Run a shell command, stream its output, and raise if it fails (stops Save & Run All)."""
    print(f'\n$ {cmd}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    print(f'[ok in {time.time() - t0:.0f}s]', flush=True)

REPO = '/kaggle/working/TrialSpec-LM'
if os.path.exists(REPO):
    run(f'git -C {REPO} pull')
else:
    run(f'git clone -b {BRANCH} https://github.com/Umairishuman/TrialSpec-LM.git {REPO}')
os.chdir(REPO)
run('git log --oneline -1')
run('bash scripts/kaggle_setup.sh')

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU (decisions log D-010)
os.environ['PYTHONUNBUFFERED'] = '1'

import torch
if not torch.cuda.is_available():
    raise RuntimeError(f'No GPU (torch {torch.__version__}). Set Session options -> Accelerator = GPU T4 x2, then Save Version again.')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

In [ ]:
run(f'bash scripts/kaggle_pipeline.sh {CONFIG} {EXTRA_ARGS}')

In [ ]:
# /kaggle/working is kept as the version's output. Download results.zip from the Output tab.
run('cd /kaggle/working && rm -f results.zip && zip -qr results.zip TrialSpec-LM/results TrialSpec-LM/outputs/*/final/training_meta.json && ls -la /kaggle/working')